# Huấn luyện YOLO11 — Nhận dạng biển số xe Việt Nam

**Notebook này chỉ là TRÌNH THỰC THI.** Toàn bộ logic nằm trong `ai/training/*.py` và
`ai/evaluation/*.py` của kho mã. Notebook không chứa logic huấn luyện — nó chỉ gọi script.
Nhờ vậy, kết quả chạy trên Colab và chạy dưới máy cục bộ là hoàn toàn giống nhau, và
mọi thay đổi tham số đều được ghi lại trong Git chứ không nằm trong một ô notebook.

---

## ⚠️ BẮT BUỘC LÀM TRƯỚC TIÊN: bật GPU

`Runtime` → `Change runtime type` → `Hardware accelerator` = **GPU** (T4 là đủ) → `Save`.

Nếu bỏ qua bước này, Colab cấp cho bạn một máy CPU và một epoch trên ~37.000 ảnh có thể
mất nhiều giờ. Ô số 2 bên dưới sẽ kiểm tra và báo lỗi rõ ràng nếu chưa có GPU.

---

## Trình tự các ô

| Ô | Việc | Ghi chú |
|---|---|---|
| 1 | Hướng dẫn (ô này) | Đọc trước khi chạy |
| 2 | Kiểm tra GPU | Dừng lại nếu không có GPU |
| 3 | Gắn Google Drive | **Không được bỏ qua** — chống mất checkpoint |
| 4 | Lấy mã nguồn | Chọn `clone` (từ Git) hoặc `drive` (tải sẵn lên Drive) |
| 5 | Cài `ultralytics` | Khoảng 1–2 phút |
| 6 | Chuẩn bị dataset | Giải nén từ Drive, kiểm tra `data.yaml` |
| 7 | Huấn luyện | Ô chạy lâu nhất; hỗ trợ `RESUME = True` |
| 8 | Đánh giá | Sinh báo cáo JSON + biểu đồ PNG |
| 9 | Xuất ONNX / OpenVINO | Cho suy luận trên CPU ở máy cục bộ |
| 10 | Lưu kết quả về Drive / tải về máy | Bước cuối |

---

## 🔌 Colab ngắt phiên giữa chừng thì làm gì?

Đây là chuyện **bình thường**, không phải sự cố. Hạ tầng đã chuẩn bị sẵn cho tình huống này:

1. Ô 3 gắn Drive, và ô 7 ghi thư mục chạy (`project`) **thẳng vào Drive**.
2. Cấu hình đặt `save_period: 10` → cứ 10 epoch lưu một checkpoint, ngoài `last.pt` lưu mỗi epoch.
3. Sau khi phiên mới khởi động: chạy lại ô 2 → 3 → 4 → 5 → 6, rồi ở ô 7 đặt `RESUME = True`
   và chạy lại. Script tự tìm `last.pt` mới nhất và huấn luyện tiếp đúng từ epoch đang dở.

**Mẹo giữ phiên sống lâu hơn:** giữ tab trình duyệt mở và hoạt động; không đóng máy;
Colab bản miễn phí giới hạn khoảng 12 giờ mỗi phiên và có thể thu hồi GPU bất cứ lúc nào.

---

## Đọc kết quả ở đâu?

* `runs/<tên_lần_chạy>/results.png` — đường cong loss và mAP theo epoch.
* `runs/<tên_lần_chạy>/weights/best.pt` — trọng số tốt nhất (được ô 10 lưu về Drive).
* `docs/reports/03-evaluation-*.json` — báo cáo đánh giá, **có tách riêng biển 1 dòng và 2 dòng**
  (yêu cầu NFR-A8 — đây là con số quan trọng nhất khi bảo vệ).
* `docs/reports/figures/` — đường cong PR, ma trận nhầm lẫn, phân bố độ trễ.

Chi tiết ý nghĩa từng tham số cấu hình: xem `ai/training/README.md`.

In [ ]:
# Cell 2 - Verify that a CUDA GPU is actually attached.
# Colab silently gives a CPU-only VM when the runtime type was never changed,
# so this check must fail loudly rather than let a multi-hour CPU run start.
!nvidia-smi || echo "nvidia-smi not available"

import torch

print()
print(f"torch            : {torch.__version__}")
print(f"CUDA available   : {torch.cuda.is_available()}")

if torch.cuda.is_available():
    properties = torch.cuda.get_device_properties(0)
    print(f"GPU              : {properties.name}")
    print(f"VRAM             : {properties.total_memory / 1024 ** 3:.1f} GB")
    print(f"Device count     : {torch.cuda.device_count()}")
    print()
    print("OK - GPU san sang. Chay tiep o so 3.")
else:
    raise RuntimeError(
        "KHONG TIM THAY GPU. Vao Runtime > Change runtime type > Hardware "
        "accelerator = GPU, luu lai, roi chay lai o nay. "
        "Huan luyen tren CPU se mat nhieu gio moi epoch."
    )

In [ ]:
# Cell 3 - Mount Google Drive.
# This is not optional: Colab reclaims the VM without warning and everything on
# the local disk disappears with it. Writing checkpoints to Drive is what makes
# a dropped session recoverable instead of a total loss.
from pathlib import Path

from google.colab import drive

drive.mount("/content/drive")

# Change this if your Drive layout differs. Everything else derives from it.
DRIVE_ROOT = Path("/content/drive/MyDrive/DATN")

DRIVE_RUNS = DRIVE_ROOT / "runs"        # training run directories (checkpoints)
DRIVE_DATA = DRIVE_ROOT / "datasets"    # dataset archive uploaded once
DRIVE_MODELS = DRIVE_ROOT / "models"    # published best.pt / exported formats
DRIVE_LOGS = DRIVE_ROOT / "logs"        # training log files

for directory in (DRIVE_ROOT, DRIVE_RUNS, DRIVE_DATA, DRIVE_MODELS, DRIVE_LOGS):
    directory.mkdir(parents=True, exist_ok=True)
    print(f"ready: {directory}")

In [ ]:
# Cell 4 - Get the source code onto the VM. Two supported modes:
#
#   SOURCE_MODE = "clone"  -> git clone from a remote repository (recommended:
#                             the run is then tied to a known commit).
#   SOURCE_MODE = "drive"  -> copy a project folder you uploaded to Drive
#                             (use this when the repo is private or has no remote).
#
# Either way the result is the same: /content/DATN contains the project and is
# the working directory, so `python -m ai.training.train` resolves correctly.
import os
import shutil
import sys
from pathlib import Path

SOURCE_MODE = "clone"  # "clone" or "drive"

REPO_URL = "https://github.com/<your-account>/<your-repo>.git"  # edit for "clone"
REPO_BRANCH = "main"
DRIVE_SOURCE = DRIVE_ROOT / "source"  # used for "drive": upload the project here

PROJECT_ROOT = Path("/content/DATN")

if PROJECT_ROOT.exists():
    print(f"Removing previous copy at {PROJECT_ROOT}")
    shutil.rmtree(PROJECT_ROOT)

if SOURCE_MODE == "clone":
    !git clone --branch {REPO_BRANCH} --depth 1 {REPO_URL} {PROJECT_ROOT}
elif SOURCE_MODE == "drive":
    if not DRIVE_SOURCE.is_dir():
        raise FileNotFoundError(
            f"Khong tim thay {DRIVE_SOURCE}. Hay tai thu muc du an len Drive "
            "(it nhat cac thu muc ai/ va notebooks/), hoac dat SOURCE_MODE='clone'."
        )
    shutil.copytree(DRIVE_SOURCE, PROJECT_ROOT)
else:
    raise ValueError(f"SOURCE_MODE must be 'clone' or 'drive', got {SOURCE_MODE!r}")

if not (PROJECT_ROOT / "ai" / "training" / "train.py").is_file():
    raise FileNotFoundError(
        f"{PROJECT_ROOT} khong chua ai/training/train.py - kiem tra lai REPO_URL "
        "hoac noi dung thu muc tren Drive."
    )

os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT))
print(f"\nWorking directory: {Path.cwd()}")
print("Available training configs:")
for config_file in sorted((PROJECT_ROOT / "ai" / "training" / "configs").glob("*.yaml")):
    print(f"  - {config_file.name}")

In [ ]:
# Cell 5 - Install the training dependencies.
# Colab already ships a CUDA build of torch, so ai/requirements.txt is NOT used
# here: it pins the CPU-only wheels for the local Windows machine and installing
# it would downgrade torch to a build with no CUDA support.
!pip install -q "ultralytics==8.4.101"

# Optional, only needed by ai/training/export.py in cell 9.
!pip install -q onnx onnxruntime onnxslim openvino

import ultralytics
import torch

ultralytics.checks()
print(f"\nultralytics {ultralytics.__version__} | torch {torch.__version__} | cuda={torch.cuda.is_available()}")

In [ ]:
# Cell 6 - Make the dataset available on the local VM disk.
#
# Read the dataset from the LOCAL disk, never directly from Drive: Drive I/O is
# slow and rate-limited, and an epoch that streams every image over it can take
# longer than the forward/backward pass itself.
#
# Expected on Drive: a single archive produced by the Phase 2 dataset pipeline,
# containing train/val/test image and label folders plus data.yaml.
import shutil
from pathlib import Path

import yaml

DATASET_ARCHIVE = DRIVE_DATA / "vn_plates_processed.zip"  # edit to match your file
LOCAL_DATASET = PROJECT_ROOT / "datasets" / "processed"

LOCAL_DATASET.parent.mkdir(parents=True, exist_ok=True)

if LOCAL_DATASET.is_dir() and any(LOCAL_DATASET.iterdir()):
    print(f"Dataset already present at {LOCAL_DATASET}; skipping extraction.")
elif DATASET_ARCHIVE.is_file():
    print(f"Extracting {DATASET_ARCHIVE} -> {LOCAL_DATASET} ...")
    shutil.unpack_archive(str(DATASET_ARCHIVE), str(LOCAL_DATASET))
    print("Done.")
else:
    raise FileNotFoundError(
        f"Khong tim thay {DATASET_ARCHIVE}.\n"
        "Cach xu ly: chay duong ong dataset o Phase 2 duoi may cuc bo, nen thu muc "
        "datasets/processed thanh .zip, tai len Drive vao thu muc tren, roi chay lai o nay."
    )

DATA_YAML = LOCAL_DATASET / "data.yaml"
if not DATA_YAML.is_file():
    candidates = sorted(LOCAL_DATASET.rglob("data.yaml"))
    if not candidates:
        raise FileNotFoundError(f"Khong tim thay data.yaml duoi {LOCAL_DATASET}")
    DATA_YAML = candidates[0]

descriptor = yaml.safe_load(DATA_YAML.read_text(encoding="utf-8"))
print(f"\ndata.yaml : {DATA_YAML}")
print(f"classes   : {descriptor.get('names')}")
for split in ("train", "val", "test"):
    print(f"{split:9s} : {descriptor.get(split, '<missing>')}")

image_count = sum(1 for _ in LOCAL_DATASET.rglob("*.jpg")) + sum(
    1 for _ in LOCAL_DATASET.rglob("*.png")
)
print(f"\nTotal images on disk: {image_count}")

In [ ]:
# Cell 7 - Train. All logic lives in ai/training/train.py; this cell only calls it.
#
# `--project` points at Drive so that checkpoints survive the VM being reclaimed.
# After a disconnect: re-run cells 2-6, set RESUME = True below, and run this cell
# again - train.py locates the newest last.pt and continues from that epoch.
CONFIG = "yolo11n_finetune.yaml"   # or yolo11n_baseline.yaml / yolo11s_escalation.yaml
RESUME = False                      # set True to continue an interrupted run

resume_flag = "--resume" if RESUME else ""

!python -m ai.training.train \
    --config {CONFIG} \
    --device auto \
    --project "{DRIVE_RUNS}" \
    --models-dir "{DRIVE_MODELS}" \
    --log-dir "{DRIVE_LOGS}" \
    {resume_flag}

In [ ]:
# Cell 8 - Evaluate on the held-out test split.
#
# --device cpu is deliberate: the deployed system runs on a CPU-only machine, so
# the latency percentiles in the report must describe that, not a Colab T4.
# Detection accuracy is identical on either device.
#
# The report separates single-line from two-line plates (NFR-A8) - that breakdown
# is the number the defence will focus on.
WEIGHTS = f"{DRIVE_MODELS}/best.pt"

!python -m ai.evaluation.evaluate \
    --weights "{WEIGHTS}" \
    --data "{DATA_YAML}" \
    --split test \
    --device cpu \
    --speed-samples 100

import json
from pathlib import Path

for report in sorted(Path("docs/reports").glob("03-evaluation-*.json")):
    print(f"\n===== {report} =====")
    payload = json.loads(report.read_text(encoding="utf-8"))
    print(json.dumps(payload["metrics_overall"], indent=2))
    print(json.dumps(payload["metrics_by_line_count"], indent=2))

In [ ]:
# Cell 9 - Export to the CPU inference formats used by the local deployment.
# Each export is loaded back and run once before being reported as successful.
!python -m ai.training.export \
    --weights "{DRIVE_MODELS}/best.pt" \
    --format all \
    --imgsz 640

In [ ]:
# Cell 10 - Persist the artefacts: copy everything worth keeping to Drive, then
# offer a direct browser download of best.pt.
import shutil
from pathlib import Path

from google.colab import files

ARTEFACT_DIR = DRIVE_MODELS
ARTEFACT_DIR.mkdir(parents=True, exist_ok=True)

# Evaluation reports and figures are deliverables; keep them off the VM disk.
for source_dir, pattern in (
    (Path("docs/reports"), "*.json"),
    (Path("docs/reports/figures"), "*.png"),
):
    if not source_dir.is_dir():
        continue
    destination = DRIVE_ROOT / source_dir
    destination.mkdir(parents=True, exist_ok=True)
    for item in source_dir.glob(pattern):
        shutil.copy2(item, destination / item.name)
        print(f"saved -> {destination / item.name}")

print("\nArtefacts on Drive:")
for item in sorted(ARTEFACT_DIR.iterdir()):
    size_mb = (
        sum(f.stat().st_size for f in item.rglob("*") if f.is_file())
        if item.is_dir()
        else item.stat().st_size
    ) / 1024 ** 2
    print(f"  {item.name:<40} {size_mb:8.2f} MB")

# Download best.pt to the local machine. Place it at <project>/models/best.pt -
# that is the default path the inference layer loads (ALPR_MODEL_PATH).
best_weights = ARTEFACT_DIR / "best.pt"
if best_weights.is_file():
    files.download(str(best_weights))
else:
    print(f"\nKhong tim thay {best_weights} - kiem tra lai o so 7 da chay xong chua.")